In [1]:
%pip install openai scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1", # Ollama
    api_key="sk-1234"  # Dummy key
)

In [3]:
# Set model parameters

SYSTEM_PROMPT = """
You a GAD-7 survey analyzer.
Your job is to extract answers for GAD-7 based on a given transcript. You are working with another chatbot that will generate followup.
The Generalized Anxiety Disorder 7-item scale (GAD-7) is a widely used self-administered diagnostic tool designed to screen for and assess the severity of generalized anxiety disorder (GAD).
GAD-7 question order:
q1: Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?
q2: Over the last two weeks, how often have you been bothered by not being able to stop or control worrying?
q3: Over the last two weeks, how often have you been bothered by worrying too much about different things?
q4: Over the last two weeks, how often have you been bothered by trouble relaxing?
q5: Over the last two weeks, how often have you been bothered by being so restless that it is hard to sit still?
q6: Over the last two weeks, how often have you been bothered by becoming easily annoyed or irritable?
q7: Over the last two weeks, how often have you been bothered by feeling afraid, as if something awful might happen?
Scale mapping:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day


Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- For each question q1-q7, determine the score 0-3 based on the user's responses. Only score questions you are confident you can answer correctly. Otherwise, do not score the question and leave that key-value pair out of your answer.
- Return ONLY valid JSON. Empty object is acceptable null.
- Do not include markdown fences.
- Do not include explanation.
- Return JSON in exactly this shape and no other text:

{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0"
}
"""
MODEL = 'qwen3.5:9b'

- Use a json dataset of conversation transcript
- Extract scores using LLM or NLP technique
- Validate whether they were correct

In [4]:
# Load the GAD dataset
import json 

with open('gad_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')

SUCCESS: Number of conversations matches number of scores


In [5]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(len(conv_train), len(_conv_test), len(score_train), len(_score_test))

7 3 7 3


In [6]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(messages: str) -> str:
    m = [Message(SYSTEM_PROMPT, 'system')]
    for message in messages:
        m.append(message)

    completion = client.chat.completions.create(
        model=MODEL,
        messages=m,
    )

    return completion.choices[0].message.content

In [7]:
print(conv_train[0])

[{'role': 'assistant', 'content': 'Hi there! I’m here to ask you a few questions about your feelings over the past two weeks. Let’s start with the first one: Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?'}, {'role': 'user', 'content': 'Well, there were a couple of moments, but nothing too serious.'}, {'role': 'assistant', 'content': 'Got it! Just to clarify, would you say that was not at all, several days, more than half the days, or nearly every day?'}, {'role': 'user', 'content': 'Hmm, maybe several days.'}, {'role': 'assistant', 'content': 'Thank you! Next question: Over the last two weeks, how often have you been bothered by not being able to stop or control worrying?'}, {'role': 'user', 'content': 'I mean, it’s come up a little, but I guess I managed it.'}, {'role': 'assistant', 'content': 'Thanks for sharing! To better gauge it, would you say it was not at all, several days, more than half the days, or nearly every day?'}, {'ro

In [ ]:
def agent_loop(full_conversation):
    result = {
        "q1": "",
        "q2": "",
        "q3": "",
        "q4": "",
        "q5": "",
        "q6": "",
        "q7": ""
    }

    def change_result(pred: str) -> None:
        # Naive parse prediction as json
        print(pred)
        new_result = json.loads(pred)
        for k, v in new_result.items():
            if k in result.keys() and v:
                result[k] = v
    
    # Simulate ask/response between chatbot and user from dataset
    for i in range(2, len(full_conversation), 2):
        # Update answers based on the answer
        pred = create_prediction(full_conversation[1:i]) # This should be range [:i] but dataset erroneously starts conversation with assistant/llm instead of user
        change_result(pred)
        # Determine if we have completed the survey
        if "" not in result.values():
            break 
    return result

scores = []
for i, c in enumerate(conv_train):
    print(f'Generating {i+1}/{len(conv_train)}')
    scores.append(agent_loop(c))
    print(scores[-1])

Generating 1/7
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0"
}
{'q1': '0', 'q2': '0', 'q3': '0', 'q4': '0', 'q5': '0', 'q6': '0', 'q7': '0'}
Generating 2/7
{
  "q1": "2",
  "q2": null,
  "q3": null,
  "q4": null,
  "q5": null,
  "q6": null,
  "q7": null
}
{
  "q1": "1"
}
{
  "q2": "1"
}
{
  "q1": "1",
  "q2": "2"
}
{
  "q2": "2",
  "q3": null
}
{
  "q1": "0",
  "q2": "3",
  "q3": "3"
}
{
  "q1": "1",
  "q2": "2",
  "q3": "1",
  "q4": "0"
}
{
  "q1": "1",
  "q2": "2",
  "q3": "1",
  "q4": "2"
}
{"q1":"1","q2":"2","q3":"1","q4":"2","q5":"1"}
{
  "q1": "1",
  "q2": "2",
  "q3": "1",
  "q4": "2",
  "q5": "3",
  "q6": null,
  "q7": null
}
{
  "q1": "0",
  "q2": "2",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0"
}
{'q1': '0', 'q2': '2', 'q3': '0', 'q4': '0', 'q5': '0', 'q6': '0', 'q7': '0'}
Generating 3/7
{
  "q1": "1",
  "q2": null,
  "q3": null,
  "q4": null,
  "q5": null,
  "q6": null,
  "q7": null
}
{
  "q1": "1"
}
{"q2": "1

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
from sklearn.metrics import mean_squared_error

def convert_scores_to_array(scores):
    return [int(s) if s else 0 for s in scores.values.flatten().tolist()]

pred_df = pd.DataFrame(scores)
train_df = pd.DataFrame(score_train)

mse = mean_squared_error(
    convert_scores_to_array(train_df),
    convert_scores_to_array(pred_df)
)
print(mse) 

diff_df = train_df.compare(pred_df, align_axis=1, keep_shape=True, keep_equal=True).rename(columns={'self': 'Expected', 'other': 'Predicted'}, level=1)
display(diff_df)